In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run /Workspace/Users/dhotepatil00@gmail.com/regis-healthcare/1_setup/utility

In [0]:
print(bronze_schema,silver_schema,gold_schema) 

In [0]:
dbutils.widgets.text("catalog","regis_healthcare","catalog")
dbutils.widgets.text("data_source","resident_feedback","data_source")

In [0]:
catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

#### Silver Processing

In [0]:
df_bronze = spark.sql(f"select * from {catalog}.{bronze_schema}.{data_source};")
display(df_bronze)
print(df_bronze.count())

In [0]:
# schema check
print(df_bronze.count())
df_bronze.printSchema()

In [0]:
df_bronze.columns

In [0]:
# drop duplicate
df_silver = df_bronze.dropDuplicates()
print(df_silver.count())

In [0]:
# feedback_id
df_silver = df_silver.withColumn(
    "feedback_id",
    F.trim(F.col("feedback_id"))
# resident_id
).withColumn(
    "resident_id",
    F.trim(F.col("resident_id"))
# facility_id
).withColumn(
    "facility_id",
    F.trim(F.col("facility_id"))
# feedback_date
).withColumn(
    "feedback_date",
    F.trim(F.col("feedback_date"))
# category
).withColumn(
    "category",
    F.trim(F.col("category"))
# rating
).withColumn(
    "rating",
    F.trim(F.col("rating"))
# comment
).withColumn(
    "comment",
    F.trim(F.col("comment"))
# submitted_by
).withColumn(
    "submitted_by",
    F.trim(F.col("submitted_by"))
# created_at
).withColumn(
    "resident_id",
    F.trim(F.col("resident_id"))
)

In [0]:
# null records count 
from pyspark.sql.functions import col,count,when
null_count = df_silver.select([count(when(col(c).isNull(),c)).alias(c)for c in df_silver.columns
                               ])
display(null_count)

In [0]:
display(df_silver)

#### Cleaning data in table

In [0]:
# feedback_id
duv = df_silver.filter(col("feedback_id").rlike("^//FKB"))
display(duv)
df_silver = df_silver.withColumn("feedback_id",when(col("feedback_id").rlike("^//FKB"),None).otherwise(col("feedback_id")))
display(df_silver)


In [0]:
# resident_id
duv = df_silver.filter(col("resident_id").rlike("^//RES"))
display(duv)
df_silver = df_silver.withColumn("resident_id",when(col("resident_id").rlike("^//RES"),None).otherwise(col("resident_id")))
display(df_silver)

In [0]:
# facility_id
duv = df_silver.filter(col("facility_id").rlike("^//FAC"))
display(duv)
df_silver = df_silver.withColumn("facility_id",when(col("facility_id").rlike("^//FAC"),None).otherwise(col("facility_id")))
display(df_silver)

In [0]:
# feedback_date
from pyspark.sql.functions import col,when
df_silver = df_silver.withColumn("feedback_date",when(~col("feedback_date").rlike("[-_=\\[\\(<\\>\\?#*~%$&@]"),None).otherwise(col("feedback_date")))

df_silver = df_silver.withColumn("feedback_date",when(col("feedback_date")==("9999-99-99"),None).otherwise(col("feedback_date")))

dup = df_silver.filter(col("feedback_date").rlike("9999-99-99"))
display(dup)

dup = df_silver.filter(~col("feedback_date").rlike("[-_=\\[\\(<\\>\\?#*~%$&@]"))
display(dup)

df_filt = df_silver.filter(~col("feedback_date").rlike("not-a-date"))

df_silver = df_silver.withColumn("feedback_date",when(~df_silver["feedback_date"].rlike("^[0-9]{4}-[0-9]{2}-[0-9]{2} [0-9]{2}:[0-9]{2}:[0-9]{2}$"),None).otherwise(col("feedback_date")))

df_filt = df_silver.filter(~df_silver["feedback_date"].rlike("^[0-9]{4}-[0-9]{2}-[0-9]{2} [0-9]{2}:[0-9]{2}:[0-9]{2}$"))
display(df_filt)

from pyspark.sql.functions import col, to_timestamp
df_silver = df_silver.withColumn(
    "feedback_date",
    to_timestamp(col("feedback_date"), "yyyy-MM-dd HH:mm:ss")  # specify format if needed
)
display(df_silver)

In [0]:
# category
from pyspark.sql.functions import col,when,upper,trim

df_silver = df_silver.withColumn("category",upper(trim(col("category"))))

dup = df_silver.groupBy("category").count()
display(dup)
df_silver = df_silver.withColumn("category",when(col("category").isNull(),"UNKNOWN").otherwise(col("category")))

from pyspark.sql.functions import col,when,upper,trim

replace_card = {"N/A" : "UNKNOWN",
"NAN" : "UNKNOWN",
"null" : "UNKNOWN",
"UNKNOWN" : "UNKNOWN",
"NONE" : "UNKNOWN",
"#N/A" : "UNKNOWN",
"NULL" : "UNKNOWN",
"" : "UNKNOWN"}

df_silver = df_silver.replace(replace_card,subset=["category"])

dup = df_silver.groupBy("category").count()
display(dup)

In [0]:
# rating
from pyspark.sql.functions import col,when,upper,trim
df_invalid = df_silver.filter(col("rating").rlike("[-_=\\[\\(<\\>\\?#*~%$&@]"))
# display(df_invalid)
df_filt = df_silver.filter(col("rating").rlike("[^a-zA-Z0-9]"))
display(df_filt)
df_silver = df_silver.withColumn("rating",when(col("rating").rlike("[^a-zA-Z0-9]"),0).otherwise(col("rating")).cast("double"))
df_invalid = df_silver.filter(col("rating").rlike("[^a-zA-Z0-9]"))
display(df_invalid)

In [0]:
# comment
from pyspark.sql.functions import col,when,initcap,trim
df_invalid = df_silver.filter(col("comment").rlike("[-_=\\[\\(<\\>\\?#*~%$&@]"))
# display(df_invalid)

df_silver = df_silver.withColumn("comment",when(col("comment").rlike("[-_=\\[\\(<\\>\\?#*~%$&@]"),"Not provide").otherwise(col("comment")))

# display(df_silver)
df_invalid = df_silver.filter(col("comment").rlike("[-_=\\[\\(<\\>\\?#*~%$&@]"))
# display(df_invalid)

dup = df_silver.groupBy("comment").count().filter(col("count")>1)
# display(dup)

reply_empty = {"null":"Not provide",
"UNKNOWN":"Not provide",
"N/A":"Not provide",
"NULL":"Not provide",
"" : "Not provide",
"NONE" : "Not provide",
"None" : "Not provide",
"Not provide" : "Not provide",
"NaN":"Not provide",
"NONE" : "Not provide",
"Good s3rvic3" : "Good Services",
"n/a"  : "Not provide",
"Exc3ll3nt s3rvic3, would r3comm3nd." : "Not provide"}
df_silver = df_silver.replace(reply_empty,subset = ["comment"])
df_silver = df_silver.withColumn("comment",initcap(trim(col("comment"))))
df_silver = df_silver.fillna({"comment":"Not provide"})
display(df_silver)


In [0]:
# submitted_by
from pyspark.sql.functions import col,when,upper,trim
df_silver = df_silver.withColumn("submitted_by",upper(trim(col("submitted_by"))))
dup = df_silver.groupBy("submitted_by").count().filter(col("count")>1)
# display(dup)

empty_cst = {"NAN" : "UNKNOWN",
"NULL" : "UNKNOWN",
"UNKNOWN" : "UNKNOWN",
"" : "UNKNOWN",
"N/A" : "UNKNOWN",
"#N/A" : "UNKNOWN",
"NONE" : "UNKNOWN",
"null" : "UNKNOWN"}

df_silver = df_silver.replace(empty_cst,subset = ["submitted_by"])
# display(df_silver)
df_silver = df_silver.fillna({"submitted_by":"UNKNOWN"})
dup = df_silver.groupBy("submitted_by").count().filter(col("count")>1)
display(dup)


In [0]:
# created_at
df_invalid = df_silver.filter(col("created_at").rlike("[-_=\\[\\(<\\>\\?#*~%$&@]"))
display(df_invalid)

####  Silver table load

In [0]:
df_silver.write\
    .format("delta")\
        .option("delta.enableChangeDataFeed","true")\
            .option("mergeSchema","true")\
                .option("overwriteSchema","true")\
            .mode("overwrite")\
               .saveAsTable(f"{catalog}.{silver_schema}.{data_source}")

dt = spark.sql(f"select * from {catalog}.{silver_schema}.{data_source};")
print(dt.count())
display(dt)

In [0]:
# load to s3
df_silver.write.format("delta")\
    .option("mergeSchema","true")\
    .option("overwriteSchema","true")\
    .mode("overwrite")\
    .partitionBy("current_date")\
    .save(f"s3://regis-healthcare/silver-clean-data/{data_source}/")